# Superstore Sales Analysis — Subqueries, CTEs & Window Functions

**Dataset:** [Superstore Dataset (Final)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final) — Kaggle

**Description:** Analyzing sales data using SQL by applying Subqueries, CTEs, and Window Functions to solve business queries.

**Engine:** SQLAlchemy

## Step 1.1: Environment Setup & Loading Raw Data

In [31]:
import pandas as pd
from sqlalchemy import create_engine, text

# Replace with your actual MySQL credentials
USER = 'root'
PASSWORD = 'root' #
HOST = 'localhost'
PORT = '3306'
DATABASE = 'superstore_db' 

# Create SQLAlchemy engine
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

# Loading the Superstore dataset into superstore_raw
CSV_PATH = "../dataset/superstore.csv" 
df = pd.read_csv(CSV_PATH, encoding="latin-1")

rename_map = {
    "Row ID": "row_id", "Order ID": "order_id", "Order Date": "order_date",
    "Ship Date": "ship_date", "Ship Mode": "ship_mode", "Customer ID": "customer_id",
    "Customer Name": "customer_name", "Segment": "segment", "Country": "country",
    "City": "city", "State": "state", "Postal Code": "postal_code", "Region": "region",
    "Product ID": "product_id", "Category": "category", "Sub-Category": "sub_category",
    "Product Name": "product_name", "Sales": "sales", "Quantity": "quantity",
    "Discount": "discount", "Profit": "profit"
}
df = df.rename(columns=rename_map)

# Insert into raw table
df.to_sql("superstore_raw", con=engine, if_exists="replace", index=False)
print(f"Loaded {len(df)} rows into superstore_raw")

Loaded 9994 rows into superstore_raw


## Step 1.2: Creating Tables and Inserting Data (Create Normalized Tables)

In [32]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS customers;"))
    conn.execute(text("DROP TABLE IF EXISTS products;"))
    conn.execute(text("DROP TABLE IF EXISTS orders;"))
    
    conn.execute(text("""
    CREATE TABLE customers (
        customer_id     VARCHAR(50) PRIMARY KEY,
        customer_name   VARCHAR(255),
        segment         VARCHAR(100)
    );
    """))

    conn.execute(text("""
    CREATE TABLE products (
        product_id      VARCHAR(50) PRIMARY KEY,
        product_name    VARCHAR(255),
        category        VARCHAR(100),
        sub_category    VARCHAR(100)
    );
    """))

    conn.execute(text("""
    CREATE TABLE orders (
        row_id          INT PRIMARY KEY,
        order_id        VARCHAR(50),
        order_date      VARCHAR(50),
        ship_date       VARCHAR(50),
        ship_mode       VARCHAR(100),
        customer_id     VARCHAR(50),
        product_id      VARCHAR(50),
        country         VARCHAR(100),
        city            VARCHAR(100),
        state           VARCHAR(100),
        postal_code     VARCHAR(50),
        region          VARCHAR(50),
        sales           DECIMAL(10, 2),
        quantity        INT,
        discount        DECIMAL(4, 2),
        profit          DECIMAL(10, 4)
    );
    """))

    # Clean insert for customers
    conn.execute(text("""
        INSERT INTO customers (customer_id, customer_name, segment)
        SELECT customer_id, MAX(customer_name), MAX(segment) 
        FROM superstore_raw GROUP BY customer_id
    """))

    # Clean insert for products
    conn.execute(text("""
        INSERT INTO products (product_id, product_name, category, sub_category)
        SELECT product_id, MAX(product_name), MAX(category), MAX(sub_category) 
        FROM superstore_raw GROUP BY product_id
    """))

    # Insert data using SELECT DISTINCT for orders
    conn.execute(text("""
        INSERT INTO orders (row_id, order_id, order_date, ship_date, ship_mode,
                             customer_id, product_id, country, city, state,
                             postal_code, region, sales, quantity, discount, profit)
        SELECT DISTINCT row_id, order_id, order_date, ship_date, ship_mode,
                         customer_id, product_id, country, city, state, postal_code,
                         region, sales, quantity, discount, profit
        FROM superstore_raw
    """))

print("Successfully created and populated normalized tables.")

Successfully created and populated normalized tables.


## Step 2.1: Orders greater than average sales (Subquery)

In [33]:
query = """
SELECT order_id, product_id, sales
FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders)
ORDER BY sales DESC;
"""
pd.read_sql_query(query, engine)

,order_id,product_id,sales
0,CA-2014-145317,TEC-MA-10002412,22638.48
1,CA-2016-118689,TEC-CO-10004722,17499.95
2,CA-2017-140151,TEC-CO-10004722,13999.96
3,CA-2017-127180,TEC-CO-10004722,11199.97
4,CA-2017-166709,TEC-CO-10004722,10499.97
...,...,...,...
2355,CA-2014-147298,FUR-CH-10004886,230.28
2356,CA-2017-161956,FUR-CH-10004886,230.28
2357,US-2014-106334,FUR-CH-10004886,230.28
2358,US-2016-168095,FUR-CH-10004886,230.28


## Step 2.2: Highest sales order for each customer (Subquery)

In [34]:
query = """
SELECT customer_id, order_id, sales
FROM orders
WHERE (customer_id, sales) IN (
    SELECT customer_id, MAX(sales) 
    FROM orders 
    GROUP BY customer_id
)
ORDER BY sales DESC;
"""
pd.read_sql_query(query, engine)

,customer_id,order_id,sales
0,SM-20320,CA-2014-145317,22638.48
1,TC-20980,CA-2016-118689,17499.95
2,RB-19360,CA-2017-140151,13999.96
3,TA-21385,CA-2017-127180,11199.97
4,HL-15040,CA-2017-166709,10499.97
...,...,...,...
790,CJ-11875,CA-2016-163951,16.52
791,MG-18205,CA-2017-115070,12.32
792,RS-19870,CA-2016-107475,9.65
793,LD-16855,CA-2016-152331,5.30


## Step 2.3: Total sales for each customer (CTE)

In [35]:
query = """
WITH CustomerTotals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT * FROM CustomerTotals
ORDER BY total_sales DESC;
"""
pd.read_sql_query(query, engine)

,customer_id,total_sales
0,SM-20320,25043.07
1,TC-20980,19052.22
2,RB-19360,15117.35
3,TA-21385,14595.62
4,AB-10105,14473.57
...,...,...
788,RS-19870,22.33
789,MG-18205,16.74
790,CJ-11875,16.52
791,LD-16855,5.30


## Step 2.4: Customers whose total sales are above average (CTE + Subquery)

In [36]:
query = """
WITH CustomerTotals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT customer_id, ROUND(total_sales, 2) AS total_sales 
FROM CustomerTotals
WHERE total_sales > (SELECT AVG(total_sales) FROM CustomerTotals)
ORDER BY total_sales DESC;
"""
pd.read_sql_query(query, engine)

,customer_id,total_sales
0,SM-20320,25043.07
1,TC-20980,19052.22
2,RB-19360,15117.35
3,TA-21385,14595.62
4,AB-10105,14473.57
...,...,...
289,JK-16120,2932.49
290,SW-20455,2921.54
291,ML-17410,2921.51
292,RD-19585,2912.90


## Step 2.5: Rank all customers based on total sales (Window Function)

In [37]:
query = """
WITH CustomerTotals AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT 
    customer_id, 
    ROUND(total_sales, 2) AS total_sales, 
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM CustomerTotals;
"""
pd.read_sql_query(query, engine)

,customer_id,total_sales,sales_rank
0,SM-20320,25043.07,1
1,TC-20980,19052.22,2
2,RB-19360,15117.35,3
3,TA-21385,14595.62,4
4,AB-10105,14473.57,5
...,...,...,...
788,RS-19870,22.33,789
789,MG-18205,16.74,790
790,CJ-11875,16.52,791
791,LD-16855,5.30,792


## Step 2.6: Assign row numbers to each order within a customer

In [38]:
query = """
SELECT 
    customer_id, 
    order_id, 
    order_date,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_sequence
FROM orders;
"""
pd.read_sql_query(query, engine)

,customer_id,order_id,order_date,order_sequence
0,AA-10315,CA-2015-121391,10/4/2015,1
1,AA-10315,CA-2016-103982,3/3/2016,2
2,AA-10315,CA-2016-103982,3/3/2016,3
3,AA-10315,CA-2016-103982,3/3/2016,4
4,AA-10315,CA-2016-103982,3/3/2016,5
...,...,...,...,...
9989,ZD-21925,CA-2016-152471,7/8/2016,5
9990,ZD-21925,CA-2016-152471,7/8/2016,6
9991,ZD-21925,CA-2014-143336,8/27/2014,7
9992,ZD-21925,CA-2014-143336,8/27/2014,8


## Step 2.7: Display top 3 customers based on total sales (Window Function)

In [39]:
query = """
WITH RankedCustomers AS (
    SELECT 
        customer_id, 
        SUM(sales) AS total_sales,
        RANK() OVER (ORDER BY SUM(sales) DESC) AS sales_rank
    FROM orders
    GROUP BY customer_id
)
SELECT customer_id, ROUND(total_sales, 2) AS total_sales, sales_rank 
FROM RankedCustomers 
WHERE sales_rank <= 3;
"""
pd.read_sql_query(query, engine)

,customer_id,total_sales,sales_rank
0,SM-20320,25043.07,1
1,TC-20980,19052.22,2
2,RB-19360,15117.35,3


## Step 3: Final Combined Query

In [40]:
final_query = """
WITH CustomerSales AS (
    SELECT 
        o.customer_id, 
        SUM(o.sales) AS total_sales
    FROM orders o
    GROUP BY o.customer_id
)
SELECT 
    c.customer_name, 
    ROUND(cs.total_sales, 2) AS total_sales,
    RANK() OVER(ORDER BY cs.total_sales DESC) AS `rank`
FROM CustomerSales cs
JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY `rank`;
"""
final_result = pd.read_sql_query(final_query, engine)
final_result

,customer_name,total_sales,rank
0,Sean Miller,25043.07,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.35,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
...,...,...,...
788,Roy Skaria,22.33,789
789,Mitch Gastineau,16.74,790
790,Carl Jackson,16.52,791
791,Lela Donovan,5.30,792
